In [ ]:
%%sql -r dataframe_1
USE DATABASE ECOMMERCE;
CREATE SCHEMA IF NOT EXISTS ECOMMERCE_DN_SCHEMA;
USE SCHEMA ECOMMERCE_DN_SCHEMA;

In [ ]:
%%sql -r dataframe_2
-- 1. Dim_Customers (Denormalized with Geography)

-- Renamed from customer_id for DW clarity
-- Flattened from Customers_Geography

CREATE OR REPLACE TABLE Customers (
    customer_key STRING PRIMARY KEY, 
    first_name STRING, 
    middle_name STRING, 
    last_name STRING, 
    suffix STRING,
    email STRING, 
    address STRING, 
    zip_code STRING, 
    city STRING,         
    state STRING,        
    country STRING,
    gender STRING, 
    birth_date DATE, 
    customer_segment STRING, 
    lifetime_value STRING,
    preferred_language STRING, 
    opt_in_email STRING, 
    phone_standard STRING, 
    alt_phone_standard STRING,
    is_active STRING,
    created_at TIMESTAMP, 
    updated_at TIMESTAMP
);

INSERT INTO Customers (
    customer_key, first_name, middle_name, last_name, suffix,
    email, address, zip_code, city, state, country,
    gender, birth_date, customer_segment, lifetime_value,
    preferred_language, opt_in_email, phone_standard, alt_phone_standard,
    is_active, created_at, updated_at
)
SELECT 
    c.customer_id AS customer_key, 
    c.first_name, c.middle_name, c.last_name, c.suffix,
    c.email, c.address, c.zip_code, g.city, g.state, c.country,
    c.gender, c.birth_date, c.customer_segment, c.lifetime_value,
    c.preferred_language, c.opt_in_email, c.phone_standard, c.alt_phone_standard,
    c.is_active, c.created_at, c.updated_at
FROM ECOMMERCE_SCHEMA.Customers c
LEFT JOIN ECOMMERCE_SCHEMA.Customers_Geography g ON c.geography_key = g.geography_key;

In [ ]:
%%sql -r dataframe_7
SELECT * FROM Customers LIMIT 10;

In [ ]:
%%sql -r dataframe_3
-- 2. Dim_Products
CREATE OR REPLACE TABLE Products (
    product_key STRING PRIMARY KEY, -- Renamed from product_id
    sku STRING, 
    mpn STRING, 
    upc STRING, 
    product_name STRING,
    brand STRING, 
    category STRING, 
    description STRING, 
    cost_price DOUBLE,
    retail_price DOUBLE, 
    weight_lbs DOUBLE,
    dimensions_inch STRING, 
    length_in DOUBLE, 
    width_in DOUBLE, 
    height_in DOUBLE,
    is_discontinued STRING, 
    supplier_id STRING,
    created_at TIMESTAMP, 
    updated_at TIMESTAMP
);


INSERT INTO Products (
    product_key, sku, mpn, upc, product_name, brand, category,
    description, cost_price, retail_price, weight_lbs, dimensions_inch,
    length_in, width_in, height_in, is_discontinued, supplier_id,
    created_at, updated_at
)
SELECT 
    product_id AS product_key,
    sku, mpn, upc, product_name, brand, category,
    description, cost_price, retail_price, weight_lbs, dimensions_inch,
    length_in, width_in, height_in, is_discontinued, supplier_id,
    created_at, updated_at
FROM ECOMMERCE_SCHEMA.Products;

In [ ]:
%%sql -r dataframe_8
SELECT * FROM Products LIMIT 10;

In [ ]:
%%sql -r dataframe_4
-- 3. Date_Table (Highly recommended placeholder for calendar attributes)
CREATE OR REPLACE TABLE Date_Table (
    date_key INT PRIMARY KEY, -- Format: YYYYMMDD
    full_date DATE,
    day_of_week INT,
    day_name STRING,
    month_number INT,
    month_name STRING,
    quarter INT,
    year INT,
    fiscal_year INT,
    fiscal_period INT
);

INSERT INTO Date_Table (
    date_key, full_date, day_of_week, day_name, 
    month_number, month_name, quarter, year, 
    fiscal_year, fiscal_period
)
WITH RECURSIVE_DATE_SEQUENCE AS (
    -- 1. Generate a continuous sequence of integers (0 to 4000+ days)
    SELECT 
        ROW_NUMBER() OVER (ORDER BY SEQ4()) - 1 AS day_offset
    FROM TABLE(GENERATOR(ROWCOUNT => 4018)) -- ~11 years of dates
),
BASE_DATES AS (
    -- 2. Establish a starting anchor date (e.g., January 1, 2020)
    SELECT 
        DATEADD(day, day_offset, '2020-01-01'::DATE) AS full_date
    FROM RECURSIVE_DATE_SEQUENCE
)
-- 3. Extract and format date components to fit the schema
SELECT
    -- date_key in YYYYMMDD format
    CAST(TO_CHAR(full_date, 'YYYYMMDD') AS INT) AS date_key,
    full_date,
    -- Day of week: Sunday (1) to Saturday (7) or Monday (1) to Sunday (7) depending on session parameters
    DAYOFWEEK(full_date) AS day_of_week,
    -- Full English name of the day (e.g., 'Monday')
    DAYNAME(full_date) AS day_name,
    MONTH(full_date) AS month_number,
    -- Full English name of the month (e.g., 'January')
    MONTHNAME(full_date) AS month_name,
    QUARTER(full_date) AS quarter,
    YEAR(full_date) AS year,
    
    -- FISCAL CALENDAR LOGIC: 
    -- Assuming a standard fiscal year that aligns with the calendar year.
    -- If your company's fiscal year starts in a different month (e.g., February), 
    -- adjust the calculation by adding/subtracting months using DATEADD.
    YEAR(full_date) AS fiscal_year,
    MONTH(full_date) AS fiscal_period

FROM BASE_DATES;

In [ ]:
%%sql -r dataframe_9
SELECT * FROM Date_Table LIMIT 10;

In [ ]:
%%sql -r dataframe_5
-- Centralized Fact Table at the Order Item Grain
CREATE OR REPLACE TABLE Sales_Invoices (
    -- Primary Keys / Line Identifier
    fact_sales_key STRING PRIMARY KEY, -- Generated composite key or UUID
    order_item_id STRING,
    order_id STRING,
    invoice_id STRING,
    invoice_number STRING,
    order_number STRING,

    -- Dimension Foreign Keys
    customer_key STRING REFERENCES Customers(customer_key),
    product_key STRING REFERENCES  Products(product_key),
    
    -- Date Dimension Keys (Smart Keys linked to Date_Table)
    order_date_key INT REFERENCES Date_Table(date_key),
    shipped_date_key INT REFERENCES Date_Table(date_key),
    invoice_date_key INT REFERENCES Date_Table(date_key),
    payment_due_date_key INT REFERENCES Date_Table(date_key),
    payment_paid_date_key INT REFERENCES Date_Table(date_key),

    -- Degenerate Dimensions (Transactional context without separate dimension tables)
    order_status STRING,
    payment_status STRING,
    payment_method STRING,
    shipping_provider STRING,
    tracking_number STRING,
    source_channel STRING,
    credit_terms STRING,
    billing_currency STRING,
    internal_gl_code STRING,

    -- Metrics & Measures (Line Level)
    quantity INT,
    unit_price DOUBLE,
    discount_applied_amount DOUBLE, -- Converted from INT to DOUBLE for exact currency math
    line_item_subtotal DOUBLE,      -- (quantity * unit_price)
    
    -- Allocated Header Level Metrics (Pro-rated or kept as-is based on reporting needs)
    allocated_tax_amount DOUBLE,
    allocated_shipping_fee DOUBLE,
    allocated_amount_paid DOUBLE,
    late_fee_charged DOUBLE,
    is_disputed BOOLEAN
);


INSERT INTO Sales_Invoices (
    fact_sales_key, order_item_id, order_id, invoice_id, invoice_number, order_number,
    customer_key, product_key,
    order_date_key, shipped_date_key, invoice_date_key, payment_due_date_key, payment_paid_date_key,
    order_status, payment_status, payment_method, shipping_provider, tracking_number,
    source_channel, credit_terms, billing_currency, internal_gl_code,
    quantity, unit_price, discount_applied_amount, line_item_subtotal,
    allocated_tax_amount, allocated_shipping_fee, allocated_amount_paid,
    late_fee_charged, is_disputed
)
SELECT 
    -- 1. Keys & IDs
    oi.order_item_id AS fact_sales_key, -- Using item ID as a unique line identifier
    oi.order_item_id,
    oi.order_id,
    i.invoice_id,
    i.invoice_number,
    o.order_number,

    -- 2. Dimension Foreign Keys
    o.customer_id AS customer_key,
    oi.product_id AS product_key,

    -- 3. Date Keys (Transformed from TIMESTAMP to YYYYMMDD integers)
    CAST(TO_CHAR(o.order_date, 'YYYYMMDD') AS INT) AS order_date_key,
    CAST(TO_CHAR(o.shipped_date, 'YYYYMMDD') AS INT) AS shipped_date_key,
    CAST(TO_CHAR(i.invoice_date, 'YYYYMMDD') AS INT) AS invoice_date_key,
    CAST(TO_CHAR(i.due_date, 'YYYYMMDD') AS INT) AS payment_due_date_key,
    CAST(TO_CHAR(i.paid_date, 'YYYYMMDD') AS INT) AS payment_paid_date_key,

    -- 4. Degenerate Dimensions (Context fields)
    o.status AS order_status,
    i.payment_status,
    COALESCE(i.payment_method, o.payment_method) AS payment_method,
    o.shipping_provider,
    o.tracking_number,
    o.source_channel,
    i.credit_terms,
    i.billing_currency,
    i.internal_gl_code,

    -- 5. Line Items Metrics
    oi.quantity,
    oi.unit_price,
    CAST(oi.discount_applied AS DOUBLE) AS discount_applied_amount,
    (oi.quantity * oi.unit_price) AS line_item_subtotal,

    -- 6. Header Metrics (Kept at line level or allocated depending on business rules)
    o.tax_amount AS allocated_tax_amount,
    o.shipping_fee AS allocated_shipping_fee,
    i.amount_paid AS allocated_amount_paid,
    i.late_fee_charged,
    COALESCE(i.is_disputed, FALSE) AS is_disputed

FROM ECOMMERCE_SCHEMA.Order_Items oi
LEFT JOIN ECOMMERCE_SCHEMA.Orders o ON oi.order_id = o.order_id
LEFT JOIN ECOMMERCE_SCHEMA.Invoices i ON o.order_id = i.order_id;

In [ ]:
%%sql -r dataframe_6
SELECT * FROM Sales_Invoices LIMIT 10;

In [ ]:
%%sql -r dataframe_10
SELECT DISTINCT year FROM ECOMMERCE.ECOMMERCE_DN_SCHEMA.DIM_DATE;